<a href="https://colab.research.google.com/github/danielkayjah123/Health-Data-HUB/blob/main/NEMS_NEW_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📊 NEMS 2025–2026 Ambulance Data Integration & Cleaning
1. Objective

The goal of this analysis is to:

- Load and combine NEMS datasets from 2025 and 2026
- Identify and resolve schema inconsistencies
- Clean and standardize categorical and textual data
- Enforce consistent data types
- Produce a clean, analysis-ready dataset

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Data Ingestion

We load data from an Excel workbook containing multiple sheets.

Approach:
- Dynamically read all sheets
- Store each as a DataFrame in a dictionary (all_dfs)
- Sanitize sheet names for consistent referencing

Outcome:
- Successfully loaded:
    - NEMS_2025
    - Compiled_NEMS_Data_2026

import ipywidgets as widgets
from IPython.display import display

# Create a Text widget for the file_path
file_path_widget = widgets.Text(
    value='/content/drive/MyDrive/Data/NEMS 25-26 Bulletin Data/MoH_NEMS Cleaned Dataset 2025.xlsx',
    placeholder='Type your file path here',
    description='File Path:',
    disabled=False,
    continuous_update=False # Update value only on blur or Enter
)

# Display the widget
display(file_path_widget)

# Define a function to load data based on the widget's value
def load_data(file_path):
    excel_file = pd.ExcelFile(file_path)
    sheet_names = excel_file.sheet_names

    # Dictionary to store all DataFrames
    all_dfs = {}

    for sheet_name in sheet_names:
        df_name = sheet_name.replace(' ', '_').replace('-', '_').replace('.', '').replace('/', '_') # Sanitize sheet name for variable name
        all_dfs[df_name] = pd.read_excel(file_path, sheet_name=sheet_name)
        print(f"DataFrame '{df_name}' loaded from sheet: {sheet_name}")
        display(all_dfs[df_name].head())
    return all_dfs

# Use the widget's value to load the data
# This will execute the load_data function once with the initial value
# You can re-run this cell after changing the widget's value to reload data.
file_path = file_path_widget.value
all_dfs = load_data(file_path)

In [3]:
dfs = ["NEMS_2025","Compiled_NEMS_Data_2026"]

### Re-initializing `all_dfs`

It appears `all_dfs` was not defined in the current session, leading to a `NameError`. This cell re-runs the data loading logic to ensure `all_dfs` is correctly populated.

In [4]:
import ipywidgets as widgets
from IPython.display import display

# Create a Text widget for the file_path
file_path_widget = widgets.Text(
    value='/content/drive/MyDrive/MOH_dataset/MoH_NEMS Cleaned Dataset 2025.xlsx',
    placeholder='Type your file path here',
    description='File Path:',
    disabled=False,
    continuous_update=False # Update value only on blur or Enter
)

# Display the widget (optional, as its value is used programmatically here)
# display(file_path_widget)

# Define a function to load data based on the widget's value
def load_data(file_path):
    excel_file = pd.ExcelFile(file_path)
    sheet_names = excel_file.sheet_names

    # Dictionary to store all DataFrames
    all_dfs = {}

    for sheet_name in sheet_names:
        df_name = sheet_name.replace(' ', '_').replace('-', '_').replace('.', '').replace('/', '_') # Sanitize sheet name for variable name
        all_dfs[df_name] = pd.read_excel(file_path, sheet_name=sheet_name)
        print(f"DataFrame '{df_name}' loaded from sheet: {sheet_name}")
        # display(all_dfs[df_name].head()) # Avoid displaying head here to prevent verbose output
    return all_dfs

# Use the widget's value to load the data
# This will execute the load_data function once with the initial value
# You can re-run this cell after changing the widget's value to reload data.
file_path = file_path_widget.value
all_dfs = load_data(file_path)
print("'all_dfs' has been re-initialized.")

DataFrame 'NEMS_2025' loaded from sheet: NEMS 2025
DataFrame 'Compiled_NEMS_Data_2026' loaded from sheet: Compiled NEMS Data 2026
DataFrame 'Analysis' loaded from sheet: Analysis
DataFrame 'Time_Analysis' loaded from sheet: Time Analysis
'all_dfs' has been re-initialized.


### 3. Initial Schema Exploration

We compare the datasets to understand structural differences.

Key Checks:
 - Column counts per dataset
- Identification of mismatched or inconsistent column names

In [5]:
print("Column counts for specified DataFrames:")
for df_name in dfs:
    if df_name in all_dfs:
        num_columns = all_dfs[df_name].shape[1]
        print(f"DataFrame '{df_name}' has {num_columns} columns.")
    else:
        print(f"DataFrame '{df_name}' not found in 'all_dfs'.")

Column counts for specified DataFrames:
DataFrame 'NEMS_2025' has 35 columns.
DataFrame 'Compiled_NEMS_Data_2026' has 35 columns.


In [6]:
import difflib

df1_name = 'NEMS_2025'
df2_name = 'Compiled_NEMS_Data_2026'

df1_cols = set(all_dfs[df1_name].columns)
df2_cols = set(all_dfs[df2_name].columns)

# Find unique columns in each DataFrame
unique_to_df1 = list(df1_cols - df2_cols)
unique_to_df2 = list(df2_cols - df1_cols)

print("\n--- Pairwise String Similarity Test for Unique Columns ---")

if not unique_to_df1 and not unique_to_df2:
    print("Both unique column sets are empty, no comparison needed.")
elif not unique_to_df1:
    print(f"'{df1_name}' has no unique columns to compare.")
    print(f"Unique columns in '{df2_name}': {unique_to_df2}")
elif not unique_to_df2:
    print(f"'{df2_name}' has no unique columns to compare.")
    print(f"Unique columns in '{df1_name}': {unique_to_df1}")
else:
    print(f"Unique columns in '{df1_name}': {unique_to_df1}")
    print(f"Unique columns in '{df2_name}': {unique_to_df2}")

    similarity_results = []
    for col1 in unique_to_df1:
        for col2 in unique_to_df2:
            # Calculate ratio using SequenceMatcher
            similarity = difflib.SequenceMatcher(None, col1, col2).ratio()
            similarity_results.append((col1, col2, similarity))

    # Sort results by similarity in descending order
    similarity_results.sort(key=lambda x: x[2], reverse=True)

    print("\nTop 10 Similar Unique Column Pairs (and their similarity score):")
    for col1, col2, sim in similarity_results[:10]:
        print(f"- '{col1}' vs '{col2}': {sim:.2f}")

    if not similarity_results:
        print("No unique column pairs to compare.")


--- Pairwise String Similarity Test for Unique Columns ---
Unique columns in 'NEMS_2025': ['AMBULANCE_STATION', 'HOSPITAL ARRIVAL TIME', 'TIME OF CALL', 'ARRIVAL AT STATION', 'Average Dispatch Time', 'DEPARTING HOSPITAL', 'TIME OF CALL RC']
Unique columns in 'Compiled_NEMS_Data_2026': ['TIME 0F CALL', 'AMBULANCE STATION', 'ARRIVAL AT STATI0N', 'Response Time (time taken from call to Arrival Target)', 'DEPARTING H0SPITAL', 'TIME 0F CALL RC', 'H0SPITAL ARRIVAL TIME']

Top 10 Similar Unique Column Pairs (and their similarity score):
- 'HOSPITAL ARRIVAL TIME' vs 'H0SPITAL ARRIVAL TIME': 0.95
- 'ARRIVAL AT STATION' vs 'ARRIVAL AT STATI0N': 0.94
- 'DEPARTING HOSPITAL' vs 'DEPARTING H0SPITAL': 0.94
- 'AMBULANCE_STATION' vs 'AMBULANCE STATION': 0.94
- 'TIME OF CALL RC' vs 'TIME 0F CALL RC': 0.93
- 'TIME OF CALL' vs 'TIME 0F CALL': 0.92
- 'TIME OF CALL' vs 'TIME 0F CALL RC': 0.81
- 'TIME OF CALL RC' vs 'TIME 0F CALL': 0.81
- 'ARRIVAL AT STATION' vs 'AMBULANCE STATION': 0.57
- 'AMBULANCE_STATIO

### 4. Detecting Column Name Inconsistencies

We used string similarity (via difflib) to identify likely matches between mismatched columns.

Key Findings:

Examples of near-identical columns with minor inconsistencies:

- 'HOSPITAL ARRIVAL TIME' ↔ 'H0SPITAL ARRIVAL TIME'
- 'DEPARTING HOSPITAL' ↔ 'DEPARTING H0SPITAL'
- 'ARRIVAL AT STATION' ↔ 'ARRIVAL AT STATI0N'
- 'TIME OF CALL' ↔ 'TIME 0F CALL'

### **Insight:**

Most inconsistencies are due to:

OCR/data entry errors (O vs 0)

Formatting differences (spaces vs underscores)


In [7]:
import difflib

print("\n--- Pairwise String Similarity Test for Unique Columns ---")

if 'unique_to_df1' in locals() and 'unique_to_df2' in locals():
    if not unique_to_df1 and not unique_to_df2:
        print("Both unique column sets are empty, no comparison needed.")
    elif not unique_to_df1:
        print(f"'{df1_name}' has no unique columns to compare.")
        print(f"Unique columns in '{df2_name}': {unique_to_df2}")
    elif not unique_to_df2:
        print(f"'{df2_name}' has no unique columns to compare.")
        print(f"Unique columns in '{df1_name}': {unique_to_df1}")
    else:
        print(f"Unique columns in '{df1_name}': {unique_to_df1}")
        print(f"Unique columns in '{df2_name}': {unique_to_df2}")

        similarity_results = []
        for col1 in unique_to_df1:
            for col2 in unique_to_df2:
                # Calculate ratio using SequenceMatcher
                similarity = difflib.SequenceMatcher(None, col1, col2).ratio()
                similarity_results.append((col1, col2, similarity))

        # Sort results by similarity in descending order
        similarity_results.sort(key=lambda x: x[2], reverse=True)

        print("\nTop 10 Similar Unique Column Pairs (and their similarity score):")
        for col1, col2, sim in similarity_results[:10]:
            print(f"- '{col1}' vs '{col2}': {sim:.2f}")

        if not similarity_results:
            print("No unique column pairs to compare.")
else:
    print("Error: 'unique_to_df1' or 'unique_to_df2' not found. Please ensure the previous comparison cell was run.")


--- Pairwise String Similarity Test for Unique Columns ---
Unique columns in 'NEMS_2025': ['AMBULANCE_STATION', 'HOSPITAL ARRIVAL TIME', 'TIME OF CALL', 'ARRIVAL AT STATION', 'Average Dispatch Time', 'DEPARTING HOSPITAL', 'TIME OF CALL RC']
Unique columns in 'Compiled_NEMS_Data_2026': ['TIME 0F CALL', 'AMBULANCE STATION', 'ARRIVAL AT STATI0N', 'Response Time (time taken from call to Arrival Target)', 'DEPARTING H0SPITAL', 'TIME 0F CALL RC', 'H0SPITAL ARRIVAL TIME']

Top 10 Similar Unique Column Pairs (and their similarity score):
- 'HOSPITAL ARRIVAL TIME' vs 'H0SPITAL ARRIVAL TIME': 0.95
- 'ARRIVAL AT STATION' vs 'ARRIVAL AT STATI0N': 0.94
- 'DEPARTING HOSPITAL' vs 'DEPARTING H0SPITAL': 0.94
- 'AMBULANCE_STATION' vs 'AMBULANCE STATION': 0.94
- 'TIME OF CALL RC' vs 'TIME 0F CALL RC': 0.93
- 'TIME OF CALL' vs 'TIME 0F CALL': 0.92
- 'TIME OF CALL' vs 'TIME 0F CALL RC': 0.81
- 'TIME OF CALL RC' vs 'TIME 0F CALL': 0.81
- 'ARRIVAL AT STATION' vs 'AMBULANCE STATION': 0.57
- 'AMBULANCE_STATIO

## Findings
- 'HOSPITAL ARRIVAL TIME' = 'H0SPITAL ARRIVAL TIME': 0.95
- 'DEPARTING HOSPITAL' = 'DEPARTING H0SPITAL': 0.94
- 'ARRIVAL AT STATION' = 'ARRIVAL AT STATI0N': 0.94
- 'AMBULANCE_STATION' = 'AMBULANCE STATION': 0.94
- 'TIME OF CALL RC' = 'TIME 0F CALL RC': 0.93
- 'TIME OF CALL' = 'TIME 0F CALL': 0.92

### 5. Assumptions (Critical)

⚠️ The following assumption was made during schema alignment:

Assumption:

> 'Response Time (time taken from call to Arrival Target)'
is equivalent to
**'Average Dispatch Time'**

Risk:
These fields may represent different operational definitions
Could introduce semantic inaccuracies

Recommendation:
### ***Validate with NEMS before relying on this mapping***

In [8]:
# Retrieve the DataFrames
df_nems_2025 = all_dfs['NEMS_2025'].copy()
df_compiled_2026 = all_dfs['Compiled_NEMS_Data_2026'].copy()

# Define the renaming dictionary for df_compiled_2026
rename_mapping = {
    'H0SPITAL ARRIVAL TIME': 'HOSPITAL ARRIVAL TIME',
    'DEPARTING H0SPITAL': 'DEPARTING HOSPITAL',
    'ARRIVAL AT STATI0N': 'ARRIVAL AT STATION',
    'AMBULANCE STATION': 'AMBULANCE_STATION',
    'TIME 0F CALL RC': 'TIME OF CALL RC',
    'TIME 0F CALL': 'TIME OF CALL',
    'Response Time (time taken from call to Arrival Target)': 'Average Dispatch Time'
}

# Rename columns in df_compiled_2026
df_compiled_2026.rename(columns=rename_mapping, inplace=True)

# Identify common columns after renaming to ensure a clean concatenation
common_cols = list(set(df_nems_2025.columns) & set(df_compiled_2026.columns))

# Select only common columns for concatenation to avoid NaN values from non-matching columns
# If there are columns unique to one DF that should be preserved, consider pd.merge or handling them specifically
df_nems_2025_aligned = df_nems_2025[common_cols]
df_compiled_2026_aligned = df_compiled_2026[common_cols]

# Concatenate the two DataFrames vertically
merged_df = pd.concat([df_nems_2025_aligned, df_compiled_2026_aligned], ignore_index=True)

print(f"Merged DataFrame created with {len(merged_df.columns)} columns and {len(merged_df)} rows.")
print("Head of the merged DataFrame:")
display(merged_df.head())

# Optional: Display info of the merged DataFrame
print("Info of the merged DataFrame:")
merged_df.info()

Merged DataFrame created with 35 columns and 21215 rows.
Head of the merged DataFrame:


,DESTINATION HOSPITAL,PHU NAME,AVERAGE TIME TAKEN FROM CALL TO DISPATCH,TYPE,TIME OF CALL,NEMS CODE,AMBULANCE_STATION,CALL_INTERVAL,PRIORITY,DISPATCH TIME,...,COMPLAINT,CONCLUSION3,ARRIVAL AT STATION,DEPART SCENE TIME,Average Dispatch Time,CONCLUSION2,AVERAGE TIME TAKEN TO REACH THE HOSPITAL,AVERAGE TIME TAKEN TO REACH THE TARGET,CALL_FROM,CONCLUSION
0,Connaught Hospital,Waterloo (BEmOC),NaN,HUB,NaN,JAN025012,W U 01,NIGHT,RED,22:35:00,...,ROAD ACCIDENT,HOSPITAL,00:42:00,11:29:00,NaN,HOSPITAL,00:31:00,00:42:00,REGULAR,HOSPITAL
1,PCMH,HASTINGS,NaN,SPOKE,17:54:00,JAN025010,W U 01,NIGHT,YELLOW,18:18:00,...,OBSTETRIC,HOSPITAL,19:30:00,18:59:00,NaN,HOSPITAL,00:25:00,00:26:00,REGULAR,HOSPITAL
2,Kabala Government Hospital,ALIKALIA,04:34:00,CHC,00:35:00,JAN025005,K I 04,MORNING,YELLOW,05:10:00,...,OBSTETRIC,HOSPITAL,13:00:00,08:15:00,NaN,HOSPITAL,03:05:00,02:40:00,REGULAR,HOSPITAL
3,UNLISTED,Talia,01:25:00,SPOKE,08:41:00,JAN025021,B H 03,MORNING,YELLOW,10:08:00,...,OBSTETRIC,OTHER MEANS OF TRANSPORTATION,NaN,NaN,NaN,ABORTED,00:00:00,NaN,REGULAR,ABORTED
4,Goderich Emergency,MAKENI Government HOSPITAL,00:32:00,HOSPITAL,16:35:00,JAN025013,B M 03,AFTERNOON,RED,17:10:00,...,BURNS,HOSPITAL,00:45:00,17:20:00,00:32:00,HOSPITAL,03:00:00,00:00:00,REGULAR,HOSPITAL


Info of the merged DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21215 entries, 0 to 21214
Data columns (total 35 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   DESTINATION HOSPITAL                        20745 non-null  object        
 1   PHU NAME                                    21215 non-null  object        
 2   AVERAGE TIME TAKEN FROM CALL TO DISPATCH    18763 non-null  object        
 3   TYPE                                        21215 non-null  object        
 4   TIME OF CALL                                21073 non-null  object        
 5   NEMS CODE                                   21215 non-null  object        
 6   AMBULANCE_STATION                           20272 non-null  object        
 7   CALL_INTERVAL                               21215 non-null  object        
 8   PRIORITY                                    21215 non-nu

In [9]:
# import re

# def sanitize_col_name(col_name):
#     """Sanitizes a column name: lowercases, replaces spaces with underscores, removes special characters."""
#     col_name = str(col_name).lower() # Ensure string and convert to lowercase
#     col_name = re.sub(r'[^a-z0-9_]+', '', col_name) # Remove special characters, keep letters, numbers, underscores
#     col_name = col_name.replace(' ', '_') # Replace spaces with underscores (if any remain)
#     col_name = col_name.strip('_') # Remove leading/trailing underscores
#     return col_name

# # Apply the sanitization function to all column names in merged_df
# original_columns = merged_df.columns.tolist()
# merged_df.columns = [sanitize_col_name(col) for col in merged_df.columns]

# print("Original Column Names:")
# for col in original_columns:
#     print(f"- {col}")

# print("\nSanitized Column Names:")
# for col in merged_df.columns:
#     print(f"- {col}")

# print("\nFirst 5 rows of the DataFrame with sanitized column names:")
# display(merged_df.head())

In [10]:
# Display the data type of the 'date' column to confirm
print(f"Current data type of 'date' column: {merged_df['Date'].dtype}")

# If a specific display format is desired for the 'date' column,
# we can convert it to string representation, but it's not enforcing the data type itself.
# For example, to display as dd/mm/yyyy string:
merged_df['date_formatted'] = merged_df['Date'].dt.strftime('%d/%m/%Y')
# print("\nFirst 5 rows with formatted date column (if applicable):")
display(merged_df[['Date', 'date_formatted']].head())
print(f"Current data type of 'date' column: {merged_df['date_formatted'].dtype}")

Current data type of 'date' column: datetime64[ns]


,Date,date_formatted
0,2025-01-01,01/01/2025
1,2025-01-01,01/01/2025
2,2025-01-01,01/01/2025
3,2025-01-01,01/01/2025
4,2025-01-01,01/01/2025


Current data type of 'date' column: object


In [11]:
display(merged_df)

,DESTINATION HOSPITAL,PHU NAME,AVERAGE TIME TAKEN FROM CALL TO DISPATCH,TYPE,TIME OF CALL,NEMS CODE,AMBULANCE_STATION,CALL_INTERVAL,PRIORITY,DISPATCH TIME,...,CONCLUSION3,ARRIVAL AT STATION,DEPART SCENE TIME,Average Dispatch Time,CONCLUSION2,AVERAGE TIME TAKEN TO REACH THE HOSPITAL,AVERAGE TIME TAKEN TO REACH THE TARGET,CALL_FROM,CONCLUSION,date_formatted
0,Connaught Hospital,Waterloo (BEmOC),NaN,HUB,NaN,JAN025012,W U 01,NIGHT,RED,22:35:00,...,HOSPITAL,00:42:00,11:29:00,NaN,HOSPITAL,00:31:00,00:42:00,REGULAR,HOSPITAL,01/01/2025
1,PCMH,HASTINGS,NaN,SPOKE,17:54:00,JAN025010,W U 01,NIGHT,YELLOW,18:18:00,...,HOSPITAL,19:30:00,18:59:00,NaN,HOSPITAL,00:25:00,00:26:00,REGULAR,HOSPITAL,01/01/2025
2,Kabala Government Hospital,ALIKALIA,04:34:00,CHC,00:35:00,JAN025005,K I 04,MORNING,YELLOW,05:10:00,...,HOSPITAL,13:00:00,08:15:00,NaN,HOSPITAL,03:05:00,02:40:00,REGULAR,HOSPITAL,01/01/2025
3,UNLISTED,Talia,01:25:00,SPOKE,08:41:00,JAN025021,B H 03,MORNING,YELLOW,10:08:00,...,OTHER MEANS OF TRANSPORTATION,NaN,NaN,NaN,ABORTED,00:00:00,NaN,REGULAR,ABORTED,01/01/2025
4,Goderich Emergency,MAKENI Government HOSPITAL,00:32:00,HOSPITAL,16:35:00,JAN025013,B M 03,AFTERNOON,RED,17:10:00,...,HOSPITAL,00:45:00,17:20:00,00:32:00,HOSPITAL,03:00:00,00:00:00,REGULAR,HOSPITAL,01/01/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21210,Unlisted,Barmoi Munu,NaN,CHC,10:23:00,JAN0261314,K A 01,MORNING,YELLOW,13:15:00,...,TREATING IN PHU,NaN,NaN,NaN,ABORTED,00:00:00,00:00:00,REGULAR,ABORTED,26/01/2026
21211,Unlisted,Kailahun Government Hospital,NaN,HOSPITAL,03:34:00,JAN026110,Unlisted,MORNING,YELLOW,NaN,...,TREATING IN PHU,NaN,NaN,NaN,ABORTED,00:00:00,00:00:00,REGULAR,QEUE MISSION,03/01/2026
21212,Unlisted,Kailahun Government Hospital,NaN,HOSPITAL,03:38:00,JAN026111,Unlisted,MORNING,YELLOW,NaN,...,TREATING IN PHU,NaN,NaN,NaN,ABORTED,00:00:00,00:00:00,REGULAR,QEUE MISSION,03/01/2026
21213,Unlisted,Kumrabai Station,NaN,SPOKE,01:38:00,JAN026150,Unlisted,MORNING,YELLOW,NaN,...,TREATING IN PHU,NaN,NaN,NaN,ABORTED,00:00:00,00:00:00,REGULAR,ABORTED,04/01/2026


### 9. Categorical Feature Identification
Approach:

Columns classified as categorical if:

- Unique values < 50
OR
Unique ratio < 5%
Output:

***Identified key categorical fields such as:***

- DESTINATION HOSPITAL
- AMBULANCE_STATION
- DISTRICT
- COMPLAINT
- PRIORITY

⚠️ Limitation:
- Heuristic-based approach
- May misclassify:
     -  IDs
    - Low frequency numerical columns

In [12]:
print("\n--- Categorical Data Check for merged_df ---")

categorical_cols = []
total_rows_df = len(merged_df)
print(f"Total rows in the DataFrame: {total_rows_df}")

for col in merged_df.columns:
    if merged_df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(merged_df[col]):
        unique_values_count = merged_df[col].nunique()
        non_null_count = merged_df[col].count()
        unique_percentage = (unique_values_count / total_rows_df) * 100 # Percentage of unique values relative to total rows

        if unique_values_count < 50 or (unique_values_count / total_rows_df) < 0.05:
            print(f"Column '{col}': Appears categorical (Type: {merged_df[col].dtype}, Non-null values: {non_null_count}, Unique values: {unique_values_count}, Unique %: {unique_percentage:.2f}%) parte")
            categorical_cols.append(col)
        else:
            print(f"Column '{col}': Appears non-categorical (Type: {merged_df[col].dtype}, Non-null values: {non_null_count}, Unique values: {unique_values_count}, Unique %: {unique_percentage:.2f}%) parte")
    else:
        non_null_count = merged_df[col].count()
        print(f"Column '{col}': Not an object/categorical type (Type: {merged_df[col].dtype}, Non-null values: {non_null_count})")

print(f"\nIdentified categorical columns: {categorical_cols}")


--- Categorical Data Check for merged_df ---
Total rows in the DataFrame: 21215
Column 'DESTINATION HOSPITAL': Appears categorical (Type: object, Non-null values: 20745, Unique values: 178, Unique %: 0.84%) parte
Column 'PHU NAME': Appears non-categorical (Type: object, Non-null values: 21215, Unique values: 1579, Unique %: 7.44%) parte
Column 'AVERAGE TIME TAKEN FROM CALL TO DISPATCH': Appears categorical (Type: object, Non-null values: 18763, Unique values: 252, Unique %: 1.19%) parte
Column 'TYPE': Appears categorical (Type: object, Non-null values: 21215, Unique values: 10, Unique %: 0.05%) parte
Column 'TIME OF CALL': Appears non-categorical (Type: object, Non-null values: 21073, Unique values: 1437, Unique %: 6.77%) parte
Column 'NEMS CODE': Appears non-categorical (Type: object, Non-null values: 21215, Unique values: 21214, Unique %: 100.00%) parte
Column 'AMBULANCE_STATION': Appears categorical (Type: object, Non-null values: 20272, Unique values: 137, Unique %: 0.65%) parte
C

/tmp/ipykernel_379/1260169422.py:8: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if merged_df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(merged_df[col]):


### 10. Refinement of Categorical Features

Removed false positives (e.g., time-based metrics incorrectly classified as categorical):

Examples removed:

- AVERAGE TIME TAKEN TO TRIAGE
- Average Dispatch Time

In [13]:
columns_to_remove = [
    'AVERAGE TIME TAKEN TO TRIAGE',
    'AVERAGE TIME TAKEN FROM CALL TO DISPATCH',
    'AVERAGE TIME TAKEN TO REACH THE TARGET',
    'AVERAGE TIME TAKEN TO BE SEEN BY CLINICIAN',
    'AVERAGE TURN AROUND TIME',
    'Average Dispatch Time',
    'AVERAGE TIME TAKEN TO REACH THE HOSPITAL',
    'CALL_INTERVAL'
]

for col in columns_to_remove:
    if col in categorical_cols:
        categorical_cols.remove(col)

print(f"Updated identified categorical columns after removing time-related columns: {categorical_cols}")

Updated identified categorical columns after removing time-related columns: ['DESTINATION HOSPITAL', 'TYPE', 'AMBULANCE_STATION', 'PRIORITY', 'STATUS', 'DECISION', 'DISTRICT', 'DESTINATION HOSPITAL TYPE', 'PATIENT_GENDER', 'COMPLAINT', 'CONCLUSION3', 'CONCLUSION2', 'CALL_FROM', 'CONCLUSION', 'date_formatted']


### 11. Categorical Data Cleaning

Key Standardization Tasks:
- 11.1 Status Cleanup
    - 'ABORTED ' → 'ABORTED'
- 11.2 Conclusion Fields
  -  'RENDEZ_VOUZ' → 'RENDEZ-VOUS'


Consolidated inconsistent labels:
- 11.3 District Standardization
    - 'WESTERN URBAN' → 'Western Area Urban'
- 11.4 Type Normalization
    - 'HUB' → 'Hub'
    - 'SPOKE' → 'Spoke'



In [14]:
print("\n--- Ranking Categorical Columns by Unique Values ---")

unique_counts = {}
for col in categorical_cols:
    if col in merged_df.columns:
        unique_counts[col] = merged_df[col].nunique()
    else:
        print(f"Warning: Column '{col}' not found in merged_df.")

# Sort the columns by unique value count in descending order
sorted_categorical_cols = sorted(unique_counts.items(), key=lambda item: item[1], reverse=True)

print("Categorical Columns Ranked by Number of Unique Values (Descending Order):")
for col, count in sorted_categorical_cols:
    print(f"- '{col}': {count} unique values")


--- Ranking Categorical Columns by Unique Values ---
Categorical Columns Ranked by Number of Unique Values (Descending Order):
- 'date_formatted': 455 unique values
- 'DESTINATION HOSPITAL': 178 unique values
- 'AMBULANCE_STATION': 137 unique values
- 'CONCLUSION3': 30 unique values
- 'DISTRICT': 18 unique values
- 'COMPLAINT': 18 unique values
- 'TYPE': 10 unique values
- 'CONCLUSION2': 7 unique values
- 'PRIORITY': 5 unique values
- 'DESTINATION HOSPITAL TYPE': 5 unique values
- 'CONCLUSION': 5 unique values
- 'CALL_FROM': 4 unique values
- 'STATUS': 3 unique values
- 'DECISION': 2 unique values
- 'PATIENT_GENDER': 2 unique values


#Categorical Columns Ranked by Number of Unique Values (Descending Order):
- 'DESTINATION HOSPITAL': 178 unique values ✅
- 'AMBULANCE_STATION': 137 unique values✅
- 'CONCLUSION3': 30 unique values ✅
- 'DISTRICT': 18 unique values ✅
- 'COMPLAINT': 18 unique values ✅
- 'TYPE': 10 unique values ✅
- 'CONCLUSION2': 7 unique values✅
- 'PRIORITY': 5 unique values✅
- 'DESTINATION HOSPITAL TYPE': 5 unique values✅
- 'CONCLUSION': 5 unique values✅
- 'CALL_FROM': 4 unique values ✅
- 'STATUS': 3 unique values ✅
- 'DECISION': 2 unique values: ✅
- 'PATIENT_GENDER': 2 unique values: ✅



Analyze Decision

In [15]:
print(merged_df['DECISION'].unique())

['AMBULANCE' 'NO']


Analyze Status

In [16]:
print(merged_df['STATUS'].unique())

['CLOSED' 'ABORTED' 'ABORTED ']


Cleaned 'ABORTED ' With ABORTED'

In [17]:
merged_df['STATUS'] = merged_df['STATUS'].replace('ABORTED ', 'ABORTED')
print(merged_df['STATUS'].unique())

['CLOSED' 'ABORTED']


CALLS FROM

In [18]:
merged_df['CALL_FROM'].unique()

array(['REGULAR', nan, ' ', 'RESCUE', 'EOC 117'], dtype=object)

CONCLUSION

In [19]:
merged_df['CONCLUSION'].unique()

array(['HOSPITAL', 'ABORTED', 'RENDEZ-VOUS', 'QEUE MISSION',
       'RENDEZ_VOUZ', nan], dtype=object)

In [20]:
merged_df['CONCLUSION'] = merged_df['CONCLUSION'].replace('RENDEZ_VOUZ', 'RENDEZ-VOUS')
print(merged_df['CONCLUSION'].unique())

['HOSPITAL' 'ABORTED' 'RENDEZ-VOUS' 'QEUE MISSION' nan]


'DESTINATION HOSPITAL TYPE'

In [21]:
print(merged_df['DESTINATION HOSPITAL TYPE'].unique())

['HOSPITAL' 'ABORTED' 'HUB' 'Hub' 'ABORTED ']


In [22]:
merged_df['DESTINATION HOSPITAL TYPE'] = merged_df['DESTINATION HOSPITAL TYPE'].replace('ABORTED ', 'ABORTED').replace('HUB', 'Hub')
print(merged_df['DESTINATION HOSPITAL TYPE'].unique())

['HOSPITAL' 'ABORTED' 'Hub']


'PRIORITY'

In [23]:
print(merged_df['PRIORITY'].unique())

['RED' 'YELLOW' 'OTHER' 'BLACK' 'GREEN']


'CONCLUSION2'

In [24]:
print(merged_df['CONCLUSION2'].unique())

['HOSPITAL' 'ABORTED' 'RENDEZ-VOUS' 'QEUE MISSION' 'RENDEZ_VOUS' 'NO NEED'
 'RENDEZ_VOUZ']


In [25]:
merged_df['CONCLUSION2'] = merged_df['CONCLUSION2'].replace({'RENDEZ_VOUS': 'RENDEZ-VOUS', 'RENDEZ_VOUZ': 'RENDEZ-VOUS'})
print(merged_df['CONCLUSION2'].unique())

['HOSPITAL' 'ABORTED' 'RENDEZ-VOUS' 'QEUE MISSION' 'NO NEED']


TYPE

In [26]:
print(merged_df['TYPE'].unique())

['HUB' 'SPOKE' 'CHC' 'HOSPITAL' 'MCHP' 'CHP' 'Hospital' 'RESCUE' 'Spoke'
 'Hub']


In [27]:
merged_df['TYPE'] = merged_df['TYPE'].replace({'HUB': 'Hub', 'SPOKE': 'Spoke', 'HOSPITAL': 'Hospital'})
print(merged_df['TYPE'].unique())

['Hub' 'Spoke' 'CHC' 'Hospital' 'MCHP' 'CHP' 'RESCUE']


COMPLAINT

In [28]:
print(merged_df['COMPLAINT'].unique())

['ROAD ACCIDENT' 'OBSTETRIC' 'BURNS' 'ABDOMINAL PAIN' 'CONSCIOUSNESS'
 'OTHER' 'TRAUMA' 'PAEDIATRIC' 'BREATHING' 'SAMPLES' 'VIOLENCE'
 'CHEST PAIN' 'ANIMAL BITE' 'SEIZURES' 'DROWNING' 'ELECTRIC SHOCK' 'MPOX'
 'SCAN']


DISTRICT

In [29]:
print(merged_df['DISTRICT'].unique())

['Western Area Rural' 'KOINADUGU' 'BONTHE' 'BOMBALI' 'BO' 'KAILAHUN'
 'KONO' 'KAMBIA' 'TONKOLILI' 'PORT LOKO' 'Western Area Urban' 'KENEMA'
 'FALABA' 'KARENE' 'PUJEHUN' 'MOYAMBA' 'WESTERN URBAN' 'WESTERN RURAL']


In [30]:
merged_df['DISTRICT'] = merged_df['DISTRICT'].replace({'WESTERN URBAN': 'Western Area Urban', 'WESTERN RURAL': 'Western Area Rural'})
print(merged_df['DISTRICT'].unique())

['Western Area Rural' 'KOINADUGU' 'BONTHE' 'BOMBALI' 'BO' 'KAILAHUN'
 'KONO' 'KAMBIA' 'TONKOLILI' 'PORT LOKO' 'Western Area Urban' 'KENEMA'
 'FALABA' 'KARENE' 'PUJEHUN' 'MOYAMBA']


CONCLUSION3

In [31]:
print(merged_df['CONCLUSION3'].unique())

['HOSPITAL' 'OTHER MEANS OF TRANSPORTATION' 'TREATED AT PHU'
 'OPERATIONAL REASONS' 'REFUSED REFERAL' 'DECEASED IN PHU'
 'DECEASED IN THE AMBULANCE' 'NO BEDS AVAILABLE' 'RENDEZ_VOUS'
 'QEUE MISSION' 'AMBULANCE CHANGED' 'NO RELATIVES'
 'DELIVERED IN THE AMBULANCE' 'NO NEED' 'DECEASED' 'ABORTED'
 'DELIEVRED IN THE PHU' 'BAD ROAD NETWORK' 'TREATED AT HOME'
 'Patient was not at target' 'AMBULANCE ON ANOTHER MISSION'
 'DECEASED AT HOME' 'OPERATIONAL REASON' 'ACCIDENT' 'WRONG PHU'
 'DELIVERED IN THE PHU' 'REFUSED REFERRAL' 'TREATING IN PHU' 'RENDEZ-VOUS'
 'REFERRAL REFUSED']


In [32]:
merged_df['CONCLUSION3'] = merged_df['CONCLUSION3'].replace({
    'REFUSED REFERAL': 'REFUSED REFERRAL',
    'RENDEZ_VOUS': 'RENDEZ-VOUS',
    'DELIEVRED IN THE PHU': 'DELIVERED IN THE PHU',
    'OPERATIONAL REASON': 'OPERATIONAL REASONS',
    'TREATING IN PHU': 'TREATED AT PHU',
    'REFERRAL REFUSED': 'REFUSED REFERRAL' # Consolidate this entry
})
print(merged_df['CONCLUSION3'].unique())

['HOSPITAL' 'OTHER MEANS OF TRANSPORTATION' 'TREATED AT PHU'
 'OPERATIONAL REASONS' 'REFUSED REFERRAL' 'DECEASED IN PHU'
 'DECEASED IN THE AMBULANCE' 'NO BEDS AVAILABLE' 'RENDEZ-VOUS'
 'QEUE MISSION' 'AMBULANCE CHANGED' 'NO RELATIVES'
 'DELIVERED IN THE AMBULANCE' 'NO NEED' 'DECEASED' 'ABORTED'
 'DELIVERED IN THE PHU' 'BAD ROAD NETWORK' 'TREATED AT HOME'
 'Patient was not at target' 'AMBULANCE ON ANOTHER MISSION'
 'DECEASED AT HOME' 'ACCIDENT' 'WRONG PHU']


In [33]:
print(merged_df['CONCLUSION3'].unique())

['HOSPITAL' 'OTHER MEANS OF TRANSPORTATION' 'TREATED AT PHU'
 'OPERATIONAL REASONS' 'REFUSED REFERRAL' 'DECEASED IN PHU'
 'DECEASED IN THE AMBULANCE' 'NO BEDS AVAILABLE' 'RENDEZ-VOUS'
 'QEUE MISSION' 'AMBULANCE CHANGED' 'NO RELATIVES'
 'DELIVERED IN THE AMBULANCE' 'NO NEED' 'DECEASED' 'ABORTED'
 'DELIVERED IN THE PHU' 'BAD ROAD NETWORK' 'TREATED AT HOME'
 'Patient was not at target' 'AMBULANCE ON ANOTHER MISSION'
 'DECEASED AT HOME' 'ACCIDENT' 'WRONG PHU']


'AMBULANCE_STATION'

In [34]:
print(merged_df['AMBULANCE_STATION'].unique())

['W U 01' 'K I 04' 'B H 03' 'B M 03' 'B O 02' 'W U 06' 'K L 02' 'W R 02'
 'K L 01' 'K O 02' 'K A 01' 'T O 01' 'P L 03' 'K L 04' 'B H 04' 'K L 05'
 'W U 05' nan 'K O 04' 'W R 01' 'T O 07' 'B M 04' 'K O 01' 'K I 01'
 'B O 01' 'K E 05' 'B M 01' 'F A 03' 'K R 01' 'F A 04' 'W U 03' 'T O 04'
 'M O 05' 'T O 08' 'K A 04' 'K O 06' 'M O 03' 'M O 06' 'M O 01' 'P L 04'
 'ISBB 02' 'K O 03' 'K O 05' 'K E 01' 'B H 01' 'P L 01' 'W R 03' 'F A 01'
 'K O 99' 'K R 04' 'B M 02' 'K I 03' 'P U 01' 'M O 02' 'F A 02' 'W U 04'
 'P U 05' 'DOC 99' 'W R 06' 'P U 03' 'P L 02' 'P L 05' 'K A 03' 'P U 04'
 'K R 02' 'K L 07' 'K L 06' 'K I 02' 'K A 02' 'P U 02' 'B O 99' 'M O 04'
 'K L 03' 'B O 03' 'B H 02' 'W R 04' 'B O 05' 'W R 07' 'K E 04' 'W R 05'
 'P U 06' 'PIH' 'K R 03' 'K E 99' 'B O 06' 'K L 99' 'K R 99' 'J MPOX'
 'MPOX' '34 MPOX' 'C MPOX' 'R MPOX' 'G MPOX' 'P MPOX' 'L MPOX' 'LAKA '
 '34 AMBULANCE' 'JUI' 'WR 03' 'CONNANGHT' 'PL 04' 'W U 07' 'P L 99'
 'K A 99' 'ISBB 01' 'MSF' 'Not a Mission' 'B M 05' 'B M 99' 'K I 

In [35]:
merged_df['AMBULANCE_STATION'] = merged_df['AMBULANCE_STATION'].str.upper().str.replace(' ', '', regex=False).str.strip()
print(merged_df['AMBULANCE_STATION'].unique())

['WU01' 'KI04' 'BH03' 'BM03' 'BO02' 'WU06' 'KL02' 'WR02' 'KL01' 'KO02'
 'KA01' 'TO01' 'PL03' 'KL04' 'BH04' 'KL05' 'WU05' nan 'KO04' 'WR01' 'TO07'
 'BM04' 'KO01' 'KI01' 'BO01' 'KE05' 'BM01' 'FA03' 'KR01' 'FA04' 'WU03'
 'TO04' 'MO05' 'TO08' 'KA04' 'KO06' 'MO03' 'MO06' 'MO01' 'PL04' 'ISBB02'
 'KO03' 'KO05' 'KE01' 'BH01' 'PL01' 'WR03' 'FA01' 'KO99' 'KR04' 'BM02'
 'KI03' 'PU01' 'MO02' 'FA02' 'WU04' 'PU05' 'DOC99' 'WR06' 'PU03' 'PL02'
 'PL05' 'KA03' 'PU04' 'KR02' 'KL07' 'KL06' 'KI02' 'KA02' 'PU02' 'BO99'
 'MO04' 'KL03' 'BO03' 'BH02' 'WR04' 'BO05' 'WR07' 'KE04' 'WR05' 'PU06'
 'PIH' 'KR03' 'KE99' 'BO06' 'KL99' 'KR99' 'JMPOX' 'MPOX' '34MPOX' 'CMPOX'
 'RMPOX' 'GMPOX' 'PMPOX' 'LMPOX' 'LAKA' '34AMBULANCE' 'JUI' 'CONNANGHT'
 'WU07' 'PL99' 'KA99' 'ISBB01' 'MSF' 'NOTAMISSION' 'BM05' 'BM99' 'KI07'
 'UNLISTED' 'TO07B' 'TO07A' 'RRT' 'FA05' 'DOC' 'KL08' 'NO' 'BO07' 'DOOPS'
 'BH99' 'KA07' 'MINIBUS' 'KL09' 'BO04' 'TO06' 'RRT1' 'KE03' 'FA06' 'RRT2'
 'RRT3' 'KL05A' 'KL05B' 'RRT-1' 'H4MSF' 'RRT-2']


In [36]:
merged_df['AMBULANCE_STATION'] = merged_df['AMBULANCE_STATION'].astype('category')
current_categories = merged_df['AMBULANCE_STATION'].cat.categories.tolist()
if 'ABORTED' not in current_categories:
    new_categories = current_categories + ['ABORTED']
    merged_df['AMBULANCE_STATION'] = merged_df['AMBULANCE_STATION'].cat.set_categories(new_categories)

merged_df['AMBULANCE_STATION'] = merged_df['AMBULANCE_STATION'].fillna('ABORTED')
print(merged_df['AMBULANCE_STATION'].unique())

['WU01', 'KI04', 'BH03', 'BM03', 'BO02', ..., 'KL05A', 'KL05B', 'RRT-1', 'H4MSF', 'RRT-2']
Length: 134
Categories (134, object): ['34AMBULANCE', '34MPOX', 'BH01', 'BH02', ..., 'WU05', 'WU06', 'WU07', 'ABORTED']


In [37]:
print("--- AMBULANCE_STATION Value Counts ---")
station_counts = merged_df['AMBULANCE_STATION'].value_counts(dropna=False)
print(station_counts)

print("\n--- AMBULANCE_STATION with Outlier Appearance (Count < 5) ---")
# Define a threshold for 'outlier' appearance (e.g., less than 5 occurrences)
outlier_threshold = 5

# Identify stations with counts below the threshold
outlier_stations = station_counts[station_counts < outlier_threshold]

if not outlier_stations.empty:
    print(outlier_stations)
else:
    print(f"No AMBULANCE_STATION values appeared less than {outlier_threshold} times.")

--- AMBULANCE_STATION Value Counts ---
AMBULANCE_STATION
ABORTED    943
WR01       777
BM01       728
WU01       627
WU05       622
          ... 
LAKA         1
PIH          1
KR99         1
MINIBUS      1
RRT          1
Name: count, Length: 134, dtype: int64

--- AMBULANCE_STATION with Outlier Appearance (Count < 5) ---
AMBULANCE_STATION
KL05A          4
DOC99          4
KO06           4
PL99           4
KL03           4
DOC            3
RRT2           3
KL05B          3
ISBB01         3
BO07           3
H4MSF          2
BH99           2
KA99           2
RRT3           2
WU07           2
KL99           2
CONNANGHT      2
34AMBULANCE    1
DOOPS          1
GMPOX          1
KE03           1
KA07           1
KI07           1
LAKA           1
PIH            1
KR99           1
MINIBUS        1
RRT            1
Name: count, dtype: int64


DESTINATION HOSPITAL

In [38]:
print(merged_df['DESTINATION HOSPITAL'].unique())

['Connaught Hospital' 'PCMH' 'Kabala Government Hospital' 'UNLISTED'
 'Goderich Emergency' 'Bo Gorverment Hospital'
 'Kailahun Government Hospital' 'Koidu Government Hospital'
 'Kambia Government Hospital' 'Magburaka Gorverment Hospital'
 'Lungi Government Hospital' 'Mattru UBC Hospital' '34 MILITARY HOOSPITAL'
 'ABORTED CALLS, NOT A MISSION' 'CHOITHRAMS MEMORIAL HOSPITAL'
 'Makeni Government Hospital' 'Kenema Government Hospital'
 "Ola During Children's Hospital" 'JOJOIMA' 'Masanga Leprosy Hospital'
 'Mobile Hospital Clinic' 'Moyamba Government Hospital'
 'Port Loko Government Hospital' 'Julius Maada Bio Paediatric Center'
 'YELE HOSPITAL' 'Njala Government. Hospital' 'LIFE CARE HOSPITAL'
 'Kissy Mental Hospital' 'Kamakwie Wesleyan Hospital' 'ECOMED'
 "Shuman's Hospital Shell" 'Lakka Government Hospital'
 'Sierra Leone - China Friendship Hospital Jui' 'New Hope Medical Centre'
 'Lumley Government Hospital' 'Pujehun Government Hospital' 'M S F Kenema'
 'G&B Hospital' 'SEGBWEMA NIXON HO

### Clean 'DESTINATION HOSPITAL' Column

In [39]:
merged_df['DESTINATION HOSPITAL'] = merged_df['DESTINATION HOSPITAL'].str.upper().str.strip()

# Define a mapping for specific standardizations after initial uppercase conversion
replace_map = {
    'BO GORVERMENT HOSPITAL': 'BO GOVERNMENT HOSPITAL',
    'MAGBURAKA GORVERMENT HOSPITAL': 'MAGBURAKA GOVERNMENT HOSPITAL',
    '34 MILITARY HOOSPITAL': '34 MILITARY HOSPITAL',
    'CHOITRAM\'S HOSPITAL': 'CHOITHRAMS MEMORIAL HOSPITAL', # Standardize apostrophe and name
    'SIERRA LEONE - CHINA FREINDSHIP HOSPITAL JUI': 'SIERRA LEONE - CHINA FRIENDSHIP HOSPITAL JUI',
    'LIFE CARE': 'LIFE CARE HOSPITAL',
    'KISSY MENTAL HOME': 'KISSY MENTAL HOSPITAL',
    'YELE': 'YELE HOSPITAL',
    'SEGBWEMA': 'SEGBWEMA NIXON HOSPITAL',
    'HOME': 'PATIENTS HOME',
    'DR. RUSSEL\'S CLINIC': 'DR. RUSSELL\'S CLINIC',
    'DR. SHUMAN\'S HOSPITAL': 'SHUMAN\'S HOSPITAL',
    'SHUMAN\'S HOSPITAL SHELL': 'SHUMAN\'S HOSPITAL', # Consolidate
    'EAZI FAMILY HOSPITAL': 'EASY FAMILY HOSPITAL',
    'PARTNERS IN HEALTH': 'PARTNER\'S IN HEALTH',
    'JUI': 'SIERRA LEONE - CHINA FRIENDSHIP HOSPITAL JUI', # Consolidate 'JUI' into full name of the hospital
    'ODCH': 'OLA DURING CHILDREN\'S HOSPITAL',
    'NEW HOPE': 'NEW HOPE MEDICAL CENTRE',
    'KAMIBIA GOVERNMENT HOSPITAL': 'KAMBIA GOVERNMENT HOSPITAL', # Correct typo
    'IGNATEDEEN HOSPITAL (GUINEA)': 'IGNATADEEN HOSPITAL (GUINEA)', # Common typo correction
    'CARE & CURE HOSPITAL': 'CARE AND CURE HOSPITAL', # Standardize ampersand
    'FSSG SCHOOL COMPUND': 'FSSG SCHOOL COMPOUND' # Correct typo
}

merged_df['DESTINATION HOSPITAL'] = merged_df['DESTINATION HOSPITAL'].replace(replace_map, regex=True)

print(merged_df['DESTINATION HOSPITAL'].unique())

['CONNAUGHT HOSPITAL' 'PCMH' 'KABALA GOVERNMENT HOSPITAL' 'UNLISTED'
 'GODERICH EMERGENCY' 'BO GOVERNMENT HOSPITAL'
 'KAILAHUN GOVERNMENT HOSPITAL' 'KOIDU GOVERNMENT HOSPITAL'
 'KAMBIA GOVERNMENT HOSPITAL' 'MAGBURAKA GOVERNMENT HOSPITAL'
 'LUNGI GOVERNMENT HOSPITAL' 'MATTRU UBC HOSPITAL' '34 MILITARY HOSPITAL'
 'ABORTED CALLS, NOT A MISSION' 'CHOITHRAMS MEMORIAL HOSPITAL'
 'MAKENI GOVERNMENT HOSPITAL' 'KENEMA GOVERNMENT HOSPITAL'
 "OLA DURING CHILDREN'S HOSPITAL" 'JOJOIMA' 'MASANGA LEPROSY HOSPITAL'
 'MOBILE HOSPITAL CLINIC' 'MOYAMBA GOVERNMENT HOSPITAL'
 'PORT LOKO GOVERNMENT HOSPITAL' 'JULIUS MAADA BIO PAEDIATRIC CENTER'
 'YELE HOSPITAL HOSPITAL' 'NJALA GOVERNMENT. HOSPITAL'
 'LIFE CARE HOSPITAL HOSPITAL' 'KISSY MENTAL HOSPITAL'
 'KAMAKWIE WESLEYAN HOSPITAL' 'ECOMED' "SHUMAN'S HOSPITAL"
 'LAKKA GOVERNMENT HOSPITAL'
 'SIERRA LEONE - CHINA FRIENDSHIP HOSPITAL SIERRA LEONE - CHINA FRIENDSHIP HOSPITAL JUI'
 'NEW HOPE MEDICAL CENTRE MEDICAL CENTRE' 'LUMLEY GOVERNMENT HOSPITAL'
 'PUJEHUN G

In [40]:
print("--- DataFrame Info ---")
merged_df.info()

print("\n--- Descriptive Statistics (Numeric Columns) ---")
# Include only numeric columns for describe
numeric_cols = merged_df.select_dtypes(include=['number']).columns
if not numeric_cols.empty:
    print(merged_df[numeric_cols].describe())
else:
    print("No numeric columns to display 5-number summary.")

print("\n--- Mode and Top Value Counts (All Columns) ---")
for col in merged_df.columns:
    print(f"\nColumn: '{col}'")
    # Get the mode and its count
    mode_series = merged_df[col].mode()
    if not mode_series.empty:
        print(f"  Mode: {mode_series.iloc[0]} (Count: {merged_df[col].value_counts().get(mode_series.iloc[0], 0)})")
    else:
        print("  Mode: N/A")

    # Get top 5 value counts for all columns, including NaN if present
    top_values = merged_df[col].value_counts(dropna=False).nlargest(5)
    if not top_values.empty:
        print("  Top 5 Value Counts:")
        for val, count in top_values.items():
            print(f"    - {val}: {count}")
    else:
        print("  No values to display.")

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21215 entries, 0 to 21214
Data columns (total 36 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   DESTINATION HOSPITAL                        20745 non-null  object        
 1   PHU NAME                                    21215 non-null  object        
 2   AVERAGE TIME TAKEN FROM CALL TO DISPATCH    18763 non-null  object        
 3   TYPE                                        21215 non-null  object        
 4   TIME OF CALL                                21073 non-null  object        
 5   NEMS CODE                                   21215 non-null  object        
 6   AMBULANCE_STATION                           21215 non-null  category      
 7   CALL_INTERVAL                               21215 non-null  object        
 8   PRIORITY                                    21215 non-null  obj

### Enforce Data Types

In [41]:
# Columns to convert to timedelta
timedelta_cols = [
    'Average Dispatch Time',
    'AVERAGE TIME TAKEN TO BE SEEN BY CLINICIAN',
    'AVERAGE TIME TAKEN TO REACH THE HOSPITAL',
    'AVERAGE TIME TAKEN TO REACH THE TARGET',
    'AVERAGE TIME TAKEN FROM CALL TO DISPATCH',
    'AVERAGE TIME TAKEN TO TRIAGE',
    'AVERAGE TURN AROUND TIME'
]

# Convert to timedelta, coercing errors to NaT (Not a Time)
for col in timedelta_cols:
    # Convert all values to string first to handle datetime.time objects and other types
    merged_df[col] = merged_df[col].astype(str)
    merged_df[col] = pd.to_timedelta(merged_df[col], errors='coerce')

# Columns to convert to category type
categorical_cols_to_convert = [
    'DISTRICT',
    'COMPLAINT',
    'CONCLUSION',
    'PRIORITY',
    'DESTINATION HOSPITAL TYPE',
    'DECISION',
    'CONCLUSION3',
    'PATIENT_GENDER',
    'CALL_FROM',
    'TYPE',
    'CALL_INTERVAL',
    'DESTINATION HOSPITAL',
    'STATUS',
    'CONCLUSION2',
    'AMBULANCE_STATION'
]

for col in categorical_cols_to_convert:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].astype('category')

# Convert 'date_formatted' to datetime
merged_df['date_formatted'] = pd.to_datetime(merged_df['date_formatted'], format='%d/%m/%Y', errors='coerce')

print("--- DataFrame Info After Type Conversion ---")
merged_df.info()

--- DataFrame Info After Type Conversion ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21215 entries, 0 to 21214
Data columns (total 36 columns):
 #   Column                                      Non-Null Count  Dtype          
---  ------                                      --------------  -----          
 0   DESTINATION HOSPITAL                        20745 non-null  category       
 1   PHU NAME                                    21215 non-null  object         
 2   AVERAGE TIME TAKEN FROM CALL TO DISPATCH    18763 non-null  timedelta64[ns]
 3   TYPE                                        21215 non-null  category       
 4   TIME OF CALL                                21073 non-null  object         
 5   NEMS CODE                                   21215 non-null  object         
 6   AMBULANCE_STATION                           21215 non-null  category       
 7   CALL_INTERVAL                               21215 non-null  category       
 8   PRIORITY                       

In [42]:
# Fill NaT values in 'AVERAGE TIME TAKEN FROM CALL TO DISPATCH' with 'ABORTED' string
# Note: This will change the column's data type from timedelta to object (string).
merged_df['AVERAGE TIME TAKEN FROM CALL TO DISPATCH'] = merged_df['AVERAGE TIME TAKEN FROM CALL TO DISPATCH'].fillna('ABORTED')

print("Unique values in 'AVERAGE TIME TAKEN FROM CALL TO DISPATCH' after filling NaNs:")
print(merged_df['AVERAGE TIME TAKEN FROM CALL TO DISPATCH'].unique())
print(f"New data type for 'AVERAGE TIME TAKEN FROM CALL TO DISPATCH': {merged_df['AVERAGE TIME TAKEN FROM CALL TO DISPATCH'].dtype}")

Unique values in 'AVERAGE TIME TAKEN FROM CALL TO DISPATCH' after filling NaNs:
['ABORTED' Timedelta('0 days 04:34:00') Timedelta('0 days 01:25:00')
 Timedelta('0 days 00:32:00') Timedelta('0 days 00:28:00')
 Timedelta('0 days 00:14:00') Timedelta('0 days 00:12:00')
 Timedelta('0 days 00:10:00') Timedelta('0 days 00:06:00')
 Timedelta('0 days 00:05:00') Timedelta('0 days 00:03:00')
 Timedelta('0 days 00:02:00') Timedelta('0 days 00:30:00')
 Timedelta('0 days 00:26:00') Timedelta('0 days 00:23:00')
 Timedelta('0 days 00:20:00') Timedelta('0 days 00:13:00')
 Timedelta('0 days 00:11:00') Timedelta('0 days 00:09:00')
 Timedelta('0 days 00:08:00') Timedelta('0 days 00:07:00')
 Timedelta('0 days 00:04:00') Timedelta('0 days 00:01:00')
 Timedelta('0 days 00:00:00') Timedelta('0 days 04:14:00')
 Timedelta('0 days 00:22:00') Timedelta('0 days 00:18:00')
 Timedelta('0 days 00:15:00') Timedelta('0 days 02:04:00')
 Timedelta('0 days 00:27:00') Timedelta('0 days 00:17:00')
 Timedelta('0 days 00:29:

In [43]:
# Fill NaN values in 'TARGET TIME' with 'ABORTED'
merged_df['TARGET TIME'] = merged_df['TARGET TIME'].fillna('ABORTED')

print("Unique values in 'TARGET TIME' after filling NaNs:")
print(merged_df['TARGET TIME'].unique())
print(f"New data type for 'TARGET TIME': {merged_df['TARGET TIME'].dtype}")

Unique values in 'TARGET TIME' after filling NaNs:
[datetime.time(23, 19) datetime.time(18, 49) datetime.time(8, 0) ...
 datetime.time(21, 3) datetime.time(5, 23) datetime.time(4, 39)]
New data type for 'TARGET TIME': object


In [44]:
merged_df['DEPART TIME'] = merged_df['DEPART TIME'].fillna('ABORTED')

print("Unique values in 'DEPART TIME' after filling NaNs:")
print(merged_df['DEPART TIME'].unique())
print(f"New data type for 'DEPART TIME': {merged_df['DEPART TIME'].dtype}")

Unique values in 'DEPART TIME' after filling NaNs:
[datetime.time(22, 37) datetime.time(18, 23) datetime.time(5, 20) ...
 datetime.time(3, 21) datetime.time(4, 52) datetime.time(0, 31)]
New data type for 'DEPART TIME': object


In [45]:
merged_df['DISPATCH TIME'] = merged_df['DISPATCH TIME'].fillna('ABORTED')

print("Unique values in 'DISPATCH TIME' after filling NaNs:")
print(merged_df['DISPATCH TIME'].unique())
print(f"New data type for 'DISPATCH TIME': {merged_df['DISPATCH TIME'].dtype}")

Unique values in 'DISPATCH TIME' after filling NaNs:
[datetime.time(22, 35) datetime.time(18, 18) datetime.time(5, 10) ...
 datetime.time(5, 9) datetime.time(5, 6) datetime.time(5, 20)]
New data type for 'DISPATCH TIME': object


In [46]:
todays_date = pd.to_datetime('today').strftime('%d-%m-%Y')

In [47]:
output_path = f'/content/drive/MyDrive/Data/NEMS 25-26 Bulletin Data/nems_basic_cleaning_ambulance_data{todays_date}.csv'
merged_df.to_csv(output_path, index=False)
print(f"DataFrame successfully saved to {output_path}")

DataFrame successfully saved to /content/drive/MyDrive/Data/NEMS 25-26 Bulletin Data/nems_basic_cleaning_ambulance_data07-05-2026.csv


In [48]:
output_path = f'/content/drive/MyDrive/Data/NEMS  Data_Updated_Clean{todays_date}.csv'
merged_df.to_csv(output_path, index=False)
print(f"DataFrame successfully saved to {output_path}")

DataFrame successfully saved to /content/drive/MyDrive/Data/NEMS  Data_Updated_Clean07-05-2026.csv
